# Milestone 5 — Final 11-class model, leaderboard estimate and submission (Session 10)

**Course:** Computer Vision and Speech Recognition — EADA · **Student:** Kristof Papp

**Code (GitHub):** REPLACE_WITH_NOTEBOOK_LINK

| what | file |
|---|---|
| this notebook (Google Colab, T4 GPU) | `notebooks/m5_final_11class.ipynb` |
| predict() on a new image | `m5_final/predict.py` |
| the leaderboard file | `m5_final/make_submission.py` → `m5_final/submission.csv` |
| final model | `m5_final/model_11class.pt` |
| leaderboard metric, thresholds | `milk10k_pipeline/evaluation.py` |

**Path A (brief):** the analysis (M2–M4) was on 3 classes, so the best architecture of Milestone 4 is
**retrained once with 11 outputs** (labels from `training_gt.csv`, same split, same recipe, new last layer).

**The official metric is not argmax.** Every probability ≥ 0.5 counts as "yes, this class", then F1 per class,
then the plain mean of the 11 F1s. With softmax the 11 probabilities share 1.0, so rare classes almost never
reach 0.5. We therefore (a) estimate the official score with that exact rule on our own **test split**, and
(b) try the brief's advanced option: one threshold per class tuned on **validation**, then a rescaling so that
each threshold sits at 0.5.

## 0. Setup (Google Colab)
**Before running:** *Runtime → Change runtime type → T4 GPU*. Then *Runtime → Run all*.
Google asks once for permission to use your Drive: everything that must survive the session
(image cache, predictions, checkpoints) is saved in `MyDrive/milk10k_project/`.

In [ ]:
import os, sys, subprocess, zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
QUICK = os.environ.get("MILK10K_QUICK") == "1"        # offline test mode only (tiny subset, 1 epoch)
REPO_URL = "https://github.com/kristofpapp330/milk10k-cv-course.git"
DATA_URL = "https://isic-archive.s3.amazonaws.com/dois/10.34970-648456/milk10k.zip"

def run(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/milk10k_project")
    repo = Path("/content/milk10k-cv-course")
    if repo.exists():
        run(f"git -C {repo} pull --ff-only")
    else:
        run(f"git clone --depth 1 {REPO_URL} {repo}")
    os.chdir(repo)
    data_root = Path("/content/milk10k")
    if not list(data_root.rglob("metadata.csv")):
        zip_path = WORK / "milk10k.zip"                   # option 1: you uploaded the zip to Drive
        if not zip_path.exists():                         # option 2: download it (about 1-2 minutes)
            zip_path = Path("/content/milk10k.zip")
            run(f"wget -q -O {zip_path} {DATA_URL}")
        print("unzipping ...")
        zipfile.ZipFile(zip_path).extractall(data_root)
    os.environ["MILK10K_DIR"] = str(next(data_root.rglob("metadata.csv")).parent)
else:                                                     # local run (testing)
    repo = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    os.chdir(repo)
    WORK = Path(os.environ.get("MILK10K_WORK", repo / "outputs" / "deep_work"))

sys.path.insert(0, str(Path.cwd()))
WORK.mkdir(parents=True, exist_ok=True)
DEEP = WORK / "outputs_deep"                              # small results -> later copied to outputs/deep in the repo
PRED = DEEP / "predictions"
CKPT = WORK / "checkpoints"                               # model weights
FIG = DEEP / "figures"
for d in (DEEP, PRED, CKPT, FIG):
    d.mkdir(parents=True, exist_ok=True)

import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from milk10k_pipeline import config, deep, evaluation as ev, mlp
from milk10k_pipeline.data_io import find_images

device = deep.get_device()
NUM_WORKERS = 2 if IN_COLAB else 0
print("device:", device, "| GPU:", torch.cuda.get_device_name(0) if device.type == "cuda" else "none")
print("data:", config.DATA_DIR, "| results:", WORK)
if IN_COLAB and device.type != "cuda":
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU, then run again.")

### The data: Milestone 1 split (by lesion), one dermoscopic image per lesion

In [ ]:
splits = {}
for name in ["train", "val", "test"]:
    df = pd.read_csv(config.SPLITS_DIR / f"{name}.csv")
    df = df[df["view"].isin(config.VIEWS)].reset_index(drop=True)
    df["y3"] = df["diagnosis_1"].map({c: i for i, c in enumerate(deep.CLASSES3)})
    df["y11"] = df["dx"].map({c: i for i, c in enumerate(deep.CLASSES11)})
    if QUICK:
        df = pd.concat([g.head(40) for _, g in df.groupby("y3")]).reset_index(drop=True)
    splits[name] = df
train, val, test = splits["train"], splits["val"], splits["test"]
assert not (set(train.lesion_id) & set(val.lesion_id)) and not (set(train.lesion_id) & set(test.lesion_id)) \
    and not (set(val.lesion_id) & set(test.lesion_id)), "lesion leakage between splits!"
print({k: len(v) for k, v in splits.items()}, "lesions")

image_paths = find_images()
all_ids = pd.concat([train, val, test])["isic_id"].tolist()
cache = deep.build_cache(all_ids, image_paths, cache_path=WORK / ("cache224_quick.npz" if QUICK else "cache224.npz"))
n_tr, n_va = len(train), len(val)
IMG = {"train": cache[:n_tr], "val": cache[n_tr:n_tr + n_va], "test": cache[n_tr + n_va:]}
Y3 = {s: splits[s]["y3"].to_numpy() for s in splits}
Y11 = {s: splits[s]["y11"].to_numpy() for s in splits}
EPOCHS = 1 if QUICK else None

---
## Common helpers (identical in the M3 and M4 notebooks)
Every deep model is scored like every earlier model: **3-class macro-F1 on the same validation lesions**.
Test predictions are saved now, but they are only looked at in the Milestone 4 scoreboard, after the
best model has been chosen on validation.

In [ ]:
REGISTRY = DEEP / "models.json"      # every deep model: its file names and validation numbers

def file_key(name):
    return name.replace("/", "-").replace(" ", "_").replace("+", "and")

def save_predictions(name, proba_val, proba_test, info, Y=None):
    Y = Y3 if Y is None else Y
    key = file_key(name)
    np.savez(PRED / f"{key}.npz", val=proba_val, test=proba_test)
    f1 = ev.macro_f1(Y["val"], proba_val.argmax(1), proba_val.shape[1])
    registry = json.loads(REGISTRY.read_text()) if REGISTRY.exists() else {}
    registry[name] = {"predictions": f"{key}.npz", "val_macro_f1": round(f1, 4), **info}
    REGISTRY.write_text(json.dumps(registry, indent=2))
    print(f"{name}: validation macro-F1 = {f1:.3f}")

def plot_history(hist, title, name):
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
    ax[0].plot(hist.epoch, hist.train_loss, "--", label="train"); ax[0].plot(hist.epoch, hist.val_loss, label="validation")
    ax[0].set_title("class-weighted cross-entropy"); ax[0].legend()
    ax[1].plot(hist.epoch, hist.val_macro_f1, color="#eb6834"); ax[1].set_title("validation macro-F1")
    ax[1].axvline(hist.val_macro_f1.idxmax() + 1, ls=":", color="gray")
    for a in ax: a.set_xlabel("epoch"); a.grid(alpha=0.3)
    plt.suptitle(title); plt.tight_layout(); plt.savefig(FIG / f"{name}.png", dpi=120); plt.show()

def finetune(name, Y=None, classes=None, tag=""):
    """Train one recipe of deep.RECIPES end to end and save its predictions + checkpoint.
    Y / classes: the labels (default: the 3 diagnosis_1 classes); tag is added to the saved name."""
    Y, classes = (Y3, deep.CLASSES3) if Y is None else (Y, classes)
    k = len(classes)
    r = deep.RECIPES[name]
    out_name = name + tag
    model = deep.build_model(r["arch"], n_classes=k, pretrained=r["pretrained"])
    deep.freeze(model, r["trainable"])
    print(f"{name}: {deep.n_trainable(model):,} trainable of {deep.n_total(model):,} parameters")
    tl = deep.make_loader(IMG["train"], Y["train"], train=True, num_workers=NUM_WORKERS)
    vl = deep.make_loader(IMG["val"], Y["val"], train=False, num_workers=NUM_WORKERS)
    te = deep.make_loader(IMG["test"], Y["test"], train=False, num_workers=NUM_WORKERS)
    start = time.time()
    model, hist, best_epoch = deep.train_model(model, r, tl, vl, Y["train"], Y["val"], k, device, epochs=EPOCHS)
    minutes = (time.time() - start) / 60
    hist.to_csv(DEEP / f"history_{file_key(out_name)}.csv", index=False)
    plot_history(hist, f"{out_name} ({minutes:.1f} min)", file_key(out_name))
    ckpt = deep.save_checkpoint(model, CKPT / f"{file_key(out_name)}.pt", r["arch"],
                                classes, r["trainable"], r["pretrained"])
    save_predictions(out_name, deep.predict_proba(model, vl, device), deep.predict_proba(model, te, device),
                     {"kind": "finetune", "arch": r["arch"], "trainable params": deep.n_trainable(model),
                      "best epoch": best_epoch, "minutes": round(minutes, 1),
                      "sec per epoch": round(hist.seconds.mean(), 1), "checkpoint": ckpt.name}, Y=Y)
    return model, hist

def frozen_head(name, arch, seeds=(0, 1, 2), Y=None, classes=None, tag=""):
    """Session 7 strategy 1 / Session 9 fair comparison: frozen backbone -> embeddings once -> linear head
    trained with the Session 6 code (class weights, Adam 1e-3, batch 64, early stopping), 3 seeds."""
    emb_file = WORK / f"embeddings_{arch}{'_quick' if QUICK else ''}.npz"
    if emb_file.exists():
        E = dict(np.load(emb_file))
    else:
        start = time.time()
        E = {s: deep.backbone_embeddings(arch, IMG[s], device, num_workers=NUM_WORKERS) for s in IMG}
        np.savez(emb_file, **E)
        print(f"embeddings computed in {time.time() - start:.0f}s: {E['train'].shape[1]} numbers per lesion")
    Y, classes = (Y3, deep.CLASSES3) if Y is None else (Y, classes)
    k, out_name = len(classes), name + tag
    runs = []
    for seed in seeds:
        head, hist, best_epoch = mlp.train_mlp(E["train"].astype(np.float32), Y["train"], E["val"].astype(np.float32),
                                               Y["val"], k, hidden=(), seed=seed, epochs=EPOCHS or 50)
        runs.append((hist["val_macro_f1"].max(), seed, head, hist))
    vals = [r[0] for r in runs]
    best_f1, best_seed, head, hist = max(runs, key=lambda r: r[0])
    full = deep.model_from_frozen_head(arch, head[0], k)
    head_prefix = ["fc"] if arch == "resnet18" else ["heads"]
    ckpt = deep.save_checkpoint(full, CKPT / f"{file_key(out_name)}.pt", arch, classes, head_prefix)
    p_val = mlp.predict_proba(head, E["val"].astype(np.float32))
    p_test = mlp.predict_proba(head, E["test"].astype(np.float32))
    save_predictions(out_name, p_val, p_test, {"kind": "frozen", "arch": arch, "trainable params": mlp.n_parameters(head),
                                           "seed val macro-F1 (3 seeds)": [round(v, 3) for v in vals],
                                           "chosen seed": best_seed, "best epoch": int(hist["val_macro_f1"].idxmax()) + 1,
                                           "checkpoint": ckpt.name}, Y=Y)
    plot_history(hist, f"{out_name} (linear head on frozen embeddings, seed {best_seed})", file_key(out_name))
    return hist

---
## 1. The best architecture of Milestone 4, retrained with 11 outputs

In [ ]:
best = json.loads((DEEP / "best_model.json").read_text())
print("best 3-class model (Milestone 4):", best["name"])
print("11 classes:", deep.CLASSES11)
print(pd.Series(Y11["train"]).map(dict(enumerate(deep.CLASSES11))).value_counts().to_string())

NAME11 = best["name"] + " (11-class)"
if best["kind"] == "finetune":
    model11, hist11 = finetune(best["name"], Y=Y11, classes=deep.CLASSES11, tag=" (11-class)")
else:
    frozen_head(best["name"], best["arch"], Y=Y11, classes=deep.CLASSES11, tag=" (11-class)")
registry = json.loads(REGISTRY.read_text())
P11 = dict(np.load(PRED / registry[NAME11]["predictions"]))

## 2. Validation: argmax macro-F1 (the course metric) and the official 0.5 rule

In [ ]:
pv, pt = P11["val"], P11["test"]
val_argmax = ev.macro_f1(Y11["val"], pv.argmax(1), 11)
val_lb, val_lb_per_class = ev.leaderboard_f1(Y11["val"], pv, 0.5)
print(f"validation macro-F1, argmax          : {val_argmax:.3f}")
print(f"validation macro-F1, official 0.5 rule: {val_lb:.3f}")
per_class = ev.per_class_table(Y11["val"], pv.argmax(1), deep.CLASSES11)
per_class["F1 at 0.5 rule"] = val_lb_per_class
per_class["max probability (mean)"] = [pv[Y11["val"] == c, c].mean() if (Y11["val"] == c).any() else np.nan
                                       for c in range(11)]
per_class.round(3)

## 3. One threshold per class (tuned on VALIDATION), checked on our TEST split
Our test split plays the role of the hidden leaderboard lesions: the thresholds are chosen on validation and
then evaluated once on test. Those test numbers are the honest estimate of the official score.

In [ ]:
thr = ev.tune_thresholds(Y11["val"], pv)
print(pd.Series(thr, index=deep.CLASSES11).to_string())
est = pd.DataFrame({
    "argmax (course metric)": [val_argmax, ev.macro_f1(Y11["test"], pt.argmax(1), 11)],
    "official rule, raw softmax ≥ 0.5": [val_lb, ev.leaderboard_f1(Y11["test"], pt, 0.5)[0]],
    "official rule, rescaled (tuned thresholds)": [ev.leaderboard_f1(Y11["val"], ev.rescale_to_half(pv, thr))[0],
                                                   ev.leaderboard_f1(Y11["test"], ev.rescale_to_half(pt, thr))[0]],
}, index=["validation (thresholds tuned here: optimistic)", "our TEST split (honest estimate)"])
est.to_csv(DEEP / "m5_leaderboard_estimate.csv")
est.round(3)

In [ ]:
lb_raw = ev.leaderboard_f1(Y11["test"], pt, 0.5)[1]
lb_tuned = ev.leaderboard_f1(Y11["test"], ev.rescale_to_half(pt, thr))[1]
x = np.arange(11)
fig, ax = plt.subplots(figsize=(12, 3.8))
ax.bar(x - 0.2, lb_raw, 0.4, color="#9ec5f0", label="raw softmax, ≥ 0.5")
ax.bar(x + 0.2, lb_tuned, 0.4, color="#2a78d6", label="rescaled with validation thresholds")
ax.set_xticks(x, [f"{c}\n(n={int((Y11['test'] == i).sum())})" for i, c in enumerate(deep.CLASSES11)], fontsize=9)
ax.set_ylabel("F1 on our test split"); ax.set_title("Official-rule F1 per class: what the thresholds change")
ax.legend(); ax.grid(axis="y", alpha=0.3); plt.tight_layout(); plt.savefig(FIG / "m5_per_class_f1.png", dpi=120); plt.show()

**Decision rule, fixed before submitting:** the submission uses the rescaled probabilities if they give the higher
official-rule macro-F1 on **validation** (where the thresholds were chosen; the test row above only checks it).

In [ ]:
USE_THRESHOLDS = bool(est.iloc[0, 2] > est.iloc[0, 1])
print("submission uses tuned thresholds:", USE_THRESHOLDS)

## 4. Save the final model (with its thresholds)

In [ ]:
FINAL = WORK / "for_repo"
(FINAL / "m5_final").mkdir(parents=True, exist_ok=True)
ck = torch.load(CKPT / registry[NAME11]["checkpoint"], weights_only=False)
ck.update({"thresholds": [float(t) for t in thr], "use_thresholds": USE_THRESHOLDS,
           "trained_on": "MILK10k dermoscopic images, Milestone 1 train split (3,143 lesions)",
           "validation_macro_f1_argmax": float(val_argmax),
           "test_estimate_official_rule": float(est.iloc[1, 2] if USE_THRESHOLDS else est.iloc[1, 1]),
           "name": NAME11})
torch.save(ck, FINAL / "m5_final" / "model_11class.pt")
print("saved", FINAL / "m5_final" / "model_11class.pt",
      f"({(FINAL / 'm5_final' / 'model_11class.pt').stat().st_size / 1e6:.1f} MB)")

## 5. (Optional here) The leaderboard file
If `MILK10k_Test_Input.zip` and `MILK10k_Test_Metadata.csv` are in `MyDrive/milk10k_project/`, the submission is
created right here. Otherwise, run `python m5_final/make_submission.py` on your laptop (instructions in the README).

In [ ]:
test_zip, test_meta = WORK / "MILK10k_Test_Input.zip", WORK / "MILK10k_Test_Metadata.csv"
if test_zip.exists() and test_meta.exists():
    sys.path.insert(0, str(Path.cwd() / "m5_final"))
    from make_submission import make_submission
    test_dir = Path("/content/MILK10k_Test_Input") if IN_COLAB else WORK / "MILK10k_Test_Input"
    if not test_dir.exists():
        zipfile.ZipFile(test_zip).extractall(test_dir.parent)
    sub = make_submission(test_dir, test_meta, FINAL / "m5_final" / "model_11class.pt",
                          FINAL / "m5_final" / "submission.csv")
    display(sub.head())
else:
    print("test files not on Drive -> create the submission on your laptop with m5_final/make_submission.py")

## 6. Package everything for the GitHub repository
One zip with the exact repository paths: unzip it into your `milk10k-cv-course` folder.

In [ ]:
import shutil
pkg = FINAL / "outputs" / "deep"
if pkg.exists():
    shutil.rmtree(pkg)
shutil.copytree(DEEP, pkg)                         # predictions, histories, scoreboard, figures, model_m4.pt
zip_out = WORK / "results_for_repo.zip"
with zipfile.ZipFile(zip_out, "w", zipfile.ZIP_DEFLATED) as z:
    for f in FINAL.rglob("*"):
        if f.is_file():
            z.write(f, f.relative_to(FINAL))
print(f"created {zip_out} ({zip_out.stat().st_size / 1e6:.0f} MB): download it from Google Drive")